
## FIM 5.2 / HV 2.2 / HAND 4.6.1.4 / ras2fim 2.0 / ripple fim30
### Spring 2025

This version started as a copy from FIM_dataset_loads_v2_1.ipynb.
</br></br>
Each step in here is not done at the same time and need not be done in specific order with
a few minor exceptions. As data comes in from HAND / Ripple, that portion of this script can be corrected
and run as needed. Each full new HAND / Ripple dataload always has minor adjustments and some steps need
to be run multiple times with either code corrections here or raw data input fixes/changes.

Ensure you the status in the Github ticket, [EPIC - HV 2.2 (FIM 5.2) - Deployment Checklist #1109]
(https://github.com/NOAA-OWP/hydrovis/issues/1109)

### Major Changes for this release
- ras2fim will stay in place as the previous 4.5.11.1 (ras2fim v2.0). However, the previous static service named \n
  ras2fim_boundaries will now be replaced with a replacement static service named "hecras_boundaries". It covers \n
  boundaries for both ras2fim v2.0 and the new Ripplle (fim_30) version.
- Ripple Fim_30 does not any data to pre-load at this time, but the data it used dynamically with hand data for \n
  regular flow processing.
- The "bridges centroids" system is being kept as is. Change it in code to use the same table as bridge_points with less columns.
- A new "Bridge Flood Threat" system. While it is dynmically processing data for each forecast dataset, the data \n
  is a "library" dataset and is being fully pre-loaded now for the entire production release.


### Important:
This not a pretty answer but is temporary. Eventually we will bring this into a more automated system 
but not now. This is run only a couple of times per year, each step one or multiple times at multipe times/dates.
So.. the value equation for this to upgrade is still low at this time.

We have different ways to load different types of data.  They don't necessarily need to be standardized.
Considering these only get run a couple of times per year and most data loads in here change significantly
from load to load.
Some of the load types used:
- fim crosswalks: uses aws s3 function of table_import_from_s3. As it is loading only data, no geo, but csv's at
a HUC level (2,158), it is pretty efficent and simple. They key is no geometries
- 
      
Questions: See Rob / Shawn.


In [ ]:
# Some or all of these may need to be run, depending on how up to date the
#  jupyter enviro is that you are running.

# Most of thse have or should be loaded into the sagemaker jupyter enviro.
# ie) (terminal window): 
# conda install python-dotenv boto3 -y

# This is required only for our SageMaker enviros. Depending on what jupyter/conda enviro
# you are running these script against, it need different pip installs. In theory, there
# shoudl be none of them, but added as conda installs in the jupyter / conda enviro.

# Cell to manually pip reload a packages that the Jupyter engine not retained (Sagemaker)
!pip install boto3
!pip install numpy
!pip install geopandas
!pip install xarray
!pip install python-dotenv
!pip install geoalchemy2

# # need to downgrade pandas and sqlaclchemy but only for our "large" sagemaker instance
!python -m pip install --upgrade 'sqlalchemy<2.0'
!python -m pip install "pandas<2.2.0"


# Rob's HV EC2

#!python -m pip install --upgrade  "sqlalchemy==1.3.24"

# !pip install s3fs # to get a newer version, old one on system
# !pip install python-dotenv
# !pip install geoalchemy2
# !pip install geopandas
# # #!pip install leafmap
# !pip install rioxarray

print("")
print("** Reload of pips done")


In [1]:
import os
import codecs
import csv
import datetime as dt
import shutil
import sys

from pathlib import Path

import boto3
import botocore
import geopandas as gpd
import json
import pandas as pd
import shapely
import sqlalchemy
import xarray as xr

# fiona = "==1.10.1"  ??  (Geopandas should take care of this)
# pyarrow = "==18.1.0" ??  (Geopandas should take care of this)
# pyogrio = "==0.8.0" ??  (Geopandas should take care of this)
# psycopg2-binary =  "==2.9.10"  versus psycop2 we load in shared_functions?


from dotenv import load_dotenv
from geopandas import GeoDataFrame
from geoalchemy2 import Geometry
from io import StringIO
from shapely import wkt
from shapely.geometry import Polygon
from sqlalchemy.exc import DataError   # yes, reduntant, fix it later
from sqlalchemy.types import Text    # yes, reduntant, fix it later
# from sqlalchemy.util import with_metaclass

gpd.options.io_engine = "pyogrio"

pd.set_option("max_info_rows", 1000000)  # override
print("option set")

print("imports loaded")


option set
imports loaded


In [2]:
# Load custom imports

sys.path.append(os.path.abspath('..'))
load_path = os.path.join(os.path.abspath('..'), "helper_functions")
print(load_path)
if load_path not in sys.path:
    sys.path.append(load_path)

import shared_functions as sf
import s3_shared_functions as s3_sf
import viz_db_ingest
from viz_classes import database
from viz_db_ingest import lambda_handler as execute_db_ingest

# sys.path.append(os.path.abspath('..'))

# import helper_functions.shared_functions as sf
# import helper_functions.s3_shared_functions as s3_sf

# from helper_functions.viz_classes import database
# from helper_functions.viz_db_ingest import lambda_handler as execute_db_ingest

print("custom imports loaded")

/home/ec2-user/SageMaker/Rob_Hanna/code/ti-v2-2-manual-dataloads/hydrovis/Core/Manual_Workflows/helper_functions


custom imports loaded


In [3]:
# Load Enviro Keys 

# Sagemaker pathing
secrets_keys_path = os.path.join(Path.home(),"SageMaker", "Secrets.env")

# EC2 pathing
# secrets_keys_path = os.path.join(Path.home(), "Documents", "Secrets.env")

print(f"secrets path are at {secrets_keys_path}")

load_dotenv(secrets_keys_path)

TI_ACCESS_KEY = os.environ['TI_ACCESS_KEY']
TI_SECRET_KEY = os.environ['TI_SECRET_KEY']
TI_TOKEN = os.environ['TI_TOKEN']

T1_VIZ_PIPELINE_ARN = os.environ['T1_VIZ_PIPELINE_ARN']
TI_VIZ_DEPLOYMENT_FIM_BUCKET = os.environ['TI_VIZ_DEPLOYMENT_FIM_BUCKET']

print("secrets loaded")

secrets path are at /home/ec2-user/SageMaker/Secrets.env
secrets loaded


In [4]:

# s3://{bucket}/fim/v5_2/hand_4_6_1_2/hand_datasets
# s3://{bucket}/fim/v5_2/hand_4_6_1_2/fim_datasets (ie. catfim, fim performance, etc)
# Previous paths (ie. 4.5.11.1), it's first level bucket was named just "fim"
# and now we added a new folder level for v5_2.  fim_datasets folder used to be called
# qa_datasets.

# we won't load this into any tables at this time
# The phrase of 5.2.0 will be embedded in config files
PUBLIC_FIM_VERSION = "5.2"
S3_FIM_VERSION_PATH =  "fim/v5_2"
HAND_MODEL_VERSION = "4.6.1.4"
S3_HAND_VERSION_PATH = "hand_4_6_1_4"

HAND_ROOT_DPATH = f"{S3_FIM_VERSION_PATH}/{S3_HAND_VERSION_PATH}"
HAND_DATASETS_DPATH = f"{HAND_ROOT_DPATH}/hand_datasets"
QA_DATASETS_DPATH = f"{HAND_ROOT_DPATH}/qa_datasets"
RAS2FIM_DATASET_DATH = f"{S3_FIM_VERSION_PATH}/ras2fim_2_0"

FIM_BUCKET = TI_VIZ_DEPLOYMENT_FIM_BUCKET
PIPELINE_ARN = T1_VIZ_PIPELINE_ARN

COLUMN_NAME_FIM_VERSION = "fim_version"
COLUMN_NAME_MODEL_VERSION = "model_version"

# Sometimes these credential values get updated. To find the latest correct values, go to your AWS Console log page and click on the "Access Key"
# link to get the latest valid set. Using the "AWS environment variables" values.
# If this is not set correctly, you will get an HTTP error 400 when you call S3 lower.
# You might also see an error of 'An error occurred (NoSuchKey) when calling the GetObject operation:
# The specified key does not exist." the creds are not correct"

HYDROVIS_CRS = "EPSG:3857"
HYDROVIS_CRS_NUMBER = "3857"

print(f"HAND_DATASETS_DPATH is {HAND_DATASETS_DPATH}")
print("Global Variables loaded")

HAND_DATASETS_DPATH is fim/v5_2/hand_4_6_1_4/hand_datasets
Global Variables loaded


<h2>1 - UPLOAD FIM4 HYDRO ID/FEATURE ID CROSSWALK</h2>

In [ ]:
FIM_CROSSWALK_FPATH = os.path.join(HAND_DATASETS_DPATH, "crosswalk_table.csv")

print(f"Getting column name from {FIM_CROSSWALK_FPATH}")

s3_client = boto3.client("s3")
data = s3_client.get_object(Bucket=FIM_BUCKET, Key=FIM_CROSSWALK_FPATH)
d_reader = csv.DictReader(codecs.getreader("utf-8")(data["Body"]))
headers = d_reader.fieldnames

header_str = "("
for header in headers:
    header_str += header
    if header in ['hand_id', 'hydro_id', 'lake_id']:
        header_str += ' integer,'
    elif header in ['branch_id', 'feature_id']:
        header_str += ' bigint,'
    else:
        header_str += ' TEXT,'
header_str = header_str[:-1] + ")"
print(header_str)

db = database(db_type="viz")
with db.get_db_connection() as conn, conn.cursor() as cur:

    print(f"Deleting/Creating derived.fim4_featureid_crosswalk using columns {header_str}")
    sql = f"DROP TABLE IF EXISTS derived.fim4_featureid_crosswalk; CREATE TABLE derived.fim4_featureid_crosswalk {header_str};"
    cur.execute(sql)
    conn.commit()

    # TODO: Nov: Drop the other 2 tables? No. ignore featureid_huc_crosswalk and featureid_huc_crosswalk_ak (not ours)
    print(f"Importing {FIM_CROSSWALK_FPATH} to derived.fim4_featureid_crosswalk")
    sql = f"""
        SELECT aws_s3.table_import_from_s3(
           'derived.fim4_featureid_crosswalk',
           '',
           '(format csv, HEADER true)',
           (SELECT aws_commons.create_s3_uri(
               '{FIM_BUCKET}',
               '{FIM_CROSSWALK_FPATH}',
               'us-east-1'
                ) AS s3_uri
            ),
            aws_commons.create_aws_credentials('{TI_ACCESS_KEY}', '{TI_SECRET_KEY}', '{TI_TOKEN}')
           );
        """
    cur.execute(sql)
    conn.commit()

    # Apr 2025: Is this used? Check db dumps as it might not take this with it. Check UAT as well.

    print(f"Adding {COLUMN_NAME_MODEL_VERSION} column to derived.fim4_featureid_crosswalk")
    sql = f"ALTER TABLE derived.fim4_featureid_crosswalk ADD COLUMN IF NOT EXISTS {COLUMN_NAME_MODEL_VERSION} text DEFAULT '{HAND_MODEL_VERSION}';"
    cur.execute(sql)
    conn.commit()

    print("Adding feature id index to derived.fim4_featureid_crosswalk")
    # Drop it already exists
    sql = "DROP INDEX IF EXISTS derived.fim4_crosswalk_feature_id"
    cur.execute(sql)
    conn.commit()
    sql = "CREATE INDEX fim4_crosswalk_feature_id ON derived.fim4_featureid_crosswalk USING btree (feature_id)"
    cur.execute(sql)
    conn.commit()

    print("Adding hydro id index to derived.fim4_featureid_crosswalk")
    # Drop it already exists
    sql = "DROP INDEX IF EXISTS derived.fim4_crosswalk_hydro_id"
    cur.execute(sql)
    conn.commit()
    sql = "CREATE INDEX fim4_crosswalk_hydro_id ON derived.fim4_featureid_crosswalk USING btree (hydro_id)"
    cur.execute(sql)
    conn.commit()

print("")
print("... Estimated time to completion is just a few mins")
print("Successfully loaded derived.fim4_featureid_crosswalk and updated it")


<h2>2 - UPDATE FIM HAND PROCESSING LAMBDA ENV VARIABLE WITH NEW FIM PREFIX</h2>

https://us-east-1.console.aws.amazon.com/lambda/home?region=us-east-1#/functions/hv-vpp-ti-viz-hand-fim-processing?tab=configure

Lambda name: hv-vpp-ti-viz-hand-fim-processing

In the Configuration Tab, click on the `Environment variables` (left menu)
- change the `FIM_VERSION` to latest public version being used. (numerics only): ie: 5.2
- change the `HAND_VERSION` to latest HAND model version being used. (numerics only): ie: 4.6.1.4

- There is an optional `Environment Variable` we can use.
    - by default, path to the hand folder is built up in lambda code to be 
        {bucket}
        /fim
        /v{FIM_VERSION changing the dot to underscore)
        /hand_{HAND_VERSION changed dots to underscores}
        /hand_datasets

    - but we can add an optional variables named `HAND_PREFIX_OVERRIDE`, which override the values at under the "fim" folder.
         resulting in:
        {bucket}
        /{HAND_PREFIX_OVERRIDE}
        (including the hand_dataset pathing)
        - This pathing should point to a set of HUC folders.

        If you don't need this override, leave this HAND_PREFIX_OVERRIDE argument out.

    - if you do use the override, it has to be from the s3 bucket all the way up to the set of huc folders.
        ie) fim/rob/hand_4_6_1_4/hand_datasets  (no slashes front or back)

    

<h2>3 - UPDATE FIM DATA PREP LAMBDA ENV VARIABLE WITH NEW FIM VERSION AND MEMORY</h2>

https://us-east-1.console.aws.amazon.com/lambda/home?region=us-east-1#/functions/hv-vpp-ti-viz-fim-data-prep?tab=code

Lambda name: <b>hv-vpp-ti-viz-hand-fim-processing</b>

In the `Configuration` Tab, click on the `Environment variables` (left menu):
- change the `FIM_VERSION` to the latest fim model version. ie) 5.2
<br><br>
<b>Then:</b> Still in the Configuration Tab, now click on the `General Configuration` (left menu), followed 
by the `edit` button on the far right side, to get into the `General Configuration` page details.
<br>Change (if they are not already there)
<br>Memory (text field) to 4096 (MB)  and
<br>Ephermeral Storage to 4096 (MB) (Apr 30, 2025: do we still need to do this?)
<br>

<h4> NOTE: When you are done, put ephermal back to 1,024 which was the original. As of Apr 30, 2025, I believe memory defaults not to 4096.</h4>
<br><br>


<h2>4 - UPDATE RAS2FIM DATA DB</h2>

As of Oct 2024, we have a new fim (hand) release covered in this file, but ras2fim does not have a new
release. ras2fim will likely be loaded as new datasets become available. 
***The FIM 5.2 version will continue to use this data and the new ripple dataset will be used along side it. Likely 6.0 as well.

We do not need to make any changes of any kinda to the ras2fim tables (geocurves or max_geocurves)

#### Check the fields though

The Fim Version field needs to be deleted

NOTE: the ras2fim boundaries tables and service is being removed and reloaded as the hecras_boundaries which
has hecras_boundaries of ras2fim and ripple in it. It will is new section below

Note: Apr 11, 2025:  ras2fim does not need to be reloaded, except
    - geocurves and max_geocurves needs the model_version changed from "2.0" to "ras2fim v2.0"


In [ ]:

# Update "geocurves" to update the "fim_version" field to "FIM 5.1.0:

# No changes required for v2_2

# print("Updating geocurves table to model_version of 2.0 to follow the new versioning system and drop fim_version")

sf.execute_sql("UPDATE ras2fim.geocurves SET model_version = 'ras2fim 2.0'", db_type="viz")

sf.execute_sql("UPDATE ras2fim.max_geocurves SET model_version = 'ras2fim 2.0'", db_type="viz")

# # Drop the "fim_version" field.
# sf.execute_sql("ALTER TABLE ras2fim.geocurves DROP COLUMN fim_version", db_type="viz")

print("Updating done for geocurves and max_geocurves")


<h2>5 - AEP FIM Pipelines</h2>

Why are these all separate (6 for conus and 6 for AK). Sometimes the lambda can get mixed up and some files
get killed while another is processing. It has to do with a S3 processing output workspace path.
Pipeline can take multiple AEP calls at the same time, just be careful to make sure you get the outputs you want.

A couple other important notes:
- These AEP configurations write data directly to the aep_fim schema in the egis RDS database.
- <b>You might need to dump the aep_fim schema after that is complete for backup / deployment into other environments.</b>

In [ ]:

# This works for all AEP calls, CONUS and AK
def get_aep_pipeline_input(stage_interval, is_alaska=False):

    ak_key = "_ak" if is_alaska else ""

    # This doesn't need to the correct actual prod_name, it is just for processing and where
    # files are temp saved in s3.. (bucket)/processing_outputs/static_nwm_aep_inundation_extent_library/rf_high_water_inundation/workspace
    # When you run more than one of these aep calls at one time, these can collide / cleanup while processing the csvs
    prod_name = f"static_nwm_aep_inundation_extent_library_{stage_interval}{ak_key}"
    config_name = f"rf_{stage_interval}_inundation{ak_key}"
    trg_table_name = f"aep_fim.rf_{stage_interval}_inundation{ak_key}"
    sql_file = f"rf_{stage_interval}_inundation{ak_key}"
    service_name = f"static_nwm_aep_inundation_extent_library{ak_key}_noaa"

    print(f"Running for product : {prod_name}")
    '''
       Example of a 2 year ak
      "product": "static_nwm_aep_inundation_extent_library_2_ak",
      "name": f"rf_2_inundation_ak",
      "target_table": f"aep_fim.rf_2_inundation_ak",
      "sql_file": f"rf_2_inundation_ak"
      "static_nwm_aep_inundation_extent_library_ak_noaa"
    '''

    '''
       Example of conus high water
      "product": "static_nwm_aep_inundation_extent_library_high_water",
      "name": f"rf_high_water_inundation",
      "target_table": f"aep_fim.rf_high_water_inundation",
      "sql_file": f"rf_high_water_inundation"
      "static_nwm_aep_inundation_extent_library_noaa"
    '''

    pipeline_input = {
      "configuration": "reference",
      "job_type": "auto",
      "data_type": "channel",
      "keep_raw": False,
      "reference_time": dt.datetime.now().strftime('%Y-%m-%d 00:00:00'),
      "configuration_data_flow": {
        "db_max_flows": [],
        "db_ingest_groups": [],
        "python_preprocessing": []
      },
      "pipeline_products": [
        {
          "product": prod_name,
          "domain": "conus",
          "configuration": "reference",
          "product_type": "fim",
          "run": True,
          "fim_configs": [
            {
              "name": config_name,
              "target_table": trg_table_name,
              "fim_type": "hand",
              "sql_file": sql_file
            }
          ],
          "services": [
              service_name
          ],
          "raster_outputs": {
            "output_bucket": "",
            "output_raster_workspaces": []
          },
          "postprocess_sql": [],
          "product_summaries": [],
          "python_preprocesing_dependent": False
        }
      ],
      "sql_rename_dict": {},
      "logging_info": {
          "Timestamp": int(dt.datetime.now().timestamp())
      }
    }
    return pipeline_input

print("function: get_aep_pipeline_input loaded")

<h2>5a - Run AEP FIM Pipelines (Non Alaska).</h2>


In [ ]:

#### 2 Year Flow (conus)
name_key = "2"
flow_name = "2"
is_ak = False
pipeline_input = get_aep_pipeline_input(flow_name, is_ak)

# notice, slightly different object name
pipeline_name = f"hv_ti_data_loads_sage_aep_{name_key}_{dt.datetime.now().strftime('%Y%m%dT%H%M')}"

step_client = boto3.client('stepfunctions')
step_client.start_execution(
    stateMachineArn = T1_VIZ_PIPELINE_ARN,
    name = pipeline_name,
    input= json.dumps(pipeline_input)
)

print(f"AEP : {flow_name} (yr) flows kicked off. Can take 15 - 45 mins.")
print(f"Step Function Pipeline : hv-vpp-ti-viz-pipeline : Pipeline run name started: {pipeline_name}")


In [ ]:

#### 5 Year Flow (conus)
name_key = "5"
flow_name = "5"
is_ak = False
pipeline_input = get_aep_pipeline_input(flow_name, is_ak)

# notice, slightly different object name
pipeline_name = f"hv_ti_data_loads_sage_aep_{name_key}_{dt.datetime.now().strftime('%Y%m%dT%H%M')}"

step_client = boto3.client('stepfunctions')
step_client.start_execution(
    stateMachineArn = T1_VIZ_PIPELINE_ARN,
    name = pipeline_name,
    input= json.dumps(pipeline_input)
)

print(f"AEP : {flow_name} (yr) flows kicked off. Can take 15 - 45 mins.")
print(f"Step Function Pipeline : hv-vpp-ti-viz-pipeline : Pipeline run name started: {pipeline_name}")


In [ ]:

#### 10 Year Flow (conus)
name_key = "10"
flow_name = "10"
is_ak = False
pipeline_input = get_aep_pipeline_input(flow_name, is_ak)

# notice, slightly different object name
pipeline_name = f"hv_ti_data_loads_sage_aep_{name_key}_{dt.datetime.now().strftime('%Y%m%dT%H%M')}"

step_client = boto3.client('stepfunctions')
step_client.start_execution(
    stateMachineArn = T1_VIZ_PIPELINE_ARN,
    name = pipeline_name,
    input= json.dumps(pipeline_input)
)

print(f"AEP : {flow_name} (yr) flows kicked off. Can take 15 - 45 mins.")
print(f"Step Function Pipeline : hv-vpp-ti-viz-pipeline : Pipeline run name started: {pipeline_name}")


In [ ]:

#### 25 Year Flow (conus)
name_key = "25"
flow_name = "25"
is_ak = False
pipeline_input = get_aep_pipeline_input(flow_name, is_ak)

# notice, slightly different object name
pipeline_name = f"hv_ti_data_loads_sage_aep_{name_key}_{dt.datetime.now().strftime('%Y%m%dT%H%M')}"

step_client = boto3.client('stepfunctions')
step_client.start_execution(
    stateMachineArn = T1_VIZ_PIPELINE_ARN,
    name = pipeline_name,
    input= json.dumps(pipeline_input)
)

print(f"AEP : {flow_name} (yr) flows kicked off. Can take 15 - 45 mins.")
print(f"Step Function Pipeline : hv-vpp-ti-viz-pipeline : Pipeline run name started: {pipeline_name}")


In [ ]:

#### 50 Year Flow  (conus)
name_key = "50"
flow_name = "50"
is_ak = False
pipeline_input = get_aep_pipeline_input(flow_name, is_ak)

# notice, slightly different object name
pipeline_name = f"hv_ti_data_loads_sage_aep_{name_key}_{dt.datetime.now().strftime('%Y%m%dT%H%M')}"

step_client = boto3.client('stepfunctions')
step_client.start_execution(
    stateMachineArn = T1_VIZ_PIPELINE_ARN,
    name = pipeline_name,
    input= json.dumps(pipeline_input)
)

print(f"AEP : {flow_name} (yr) flows kicked off. Can take 15 - 45 mins.")
print(f"Step Function Pipeline : hv-vpp-ti-viz-pipeline : Pipeline run name started: {pipeline_name}")


In [ ]:

#### HW (High Water) Flow  (conus)
name_key = "hw"
flow_name = "high_water"
is_ak = False
pipeline_input = get_aep_pipeline_input(flow_name, is_ak)

# notice, slightly different object name
pipeline_name = f"hv_ti_data_loads_sage_aep_{name_key}_{dt.datetime.now().strftime('%Y%m%dT%H%M')}"

# print(pipeline_input)
step_client = boto3.client('stepfunctions')
step_client.start_execution(
     stateMachineArn = T1_VIZ_PIPELINE_ARN,
     name = pipeline_name,
     input= json.dumps(pipeline_input)
)

print(f"AEP : {flow_name} (yr) flows kicked off. Can take 15 - 45 mins.")
print(f"Step Function Pipeline : hv-vpp-ti-viz-pipeline : Pipeline run name started: {pipeline_name}")
print("")

<h2>5b - Run AEP FIM Pipelines (Alaska).</h2
A couple other important notes:
- These AEP configurations write data directly to the aep_fim schema in the egis RDS database, instead of the viz database.
- <b>You'll need to dump the aep_fim schema after that is complete for backup / deployment into other environments.</b>

***Note: You can start each of these 6 one right after the other. Maybe someday we can create a block that just starts all 6 at once.***


In [ ]:

#### 2 Year Flow for Alaska

name_key = "2_ak"
flow_name = "2"
is_ak = True
pipeline_input = get_aep_pipeline_input(flow_name, is_ak)

# notice, slightly different object name
pipeline_name = f"hv_ti_data_loads_sage_aep_{name_key}_{dt.datetime.now().strftime('%Y%m%dT%H%M')}"

step_client = boto3.client('stepfunctions')
step_client.start_execution(
    stateMachineArn = T1_VIZ_PIPELINE_ARN,
    name = pipeline_name,
    input= json.dumps(pipeline_input)
)

print(f"AEP : {flow_name} (yr) flows kicked off. Can take 15 - 45 mins.")
print(f"Step Function Pipeline : hv-vpp-ti-viz-pipeline : Pipeline run name started: {pipeline_name}")


In [ ]:

#### 5 Year Flow for Alaska

name_key = "5_ak"
flow_name = "5"
is_ak = True
pipeline_input = get_aep_pipeline_input(flow_name, is_ak)

# notice, slightly different object name
pipeline_name = f"hv_ti_data_loads_sage_aep_{name_key}_{dt.datetime.now().strftime('%Y%m%dT%H%M')}"

step_client = boto3.client('stepfunctions')
step_client.start_execution(
    stateMachineArn = T1_VIZ_PIPELINE_ARN,
    name = pipeline_name,
    input= json.dumps(pipeline_input)
)

print(f"AEP : {flow_name} (yr) flows kicked off. Can take 15 - 45 mins.")
print(f"Step Function Pipeline : hv-vpp-ti-viz-pipeline : Pipeline run name started: {pipeline_name}")


In [ ]:

#### 10 Year Flow for Alaska

name_key = "10_ak"
flow_name = "10"
is_ak = True
pipeline_input = get_aep_pipeline_input(flow_name, is_ak)

# notice, slightly different object name
pipeline_name = f"hv_ti_data_loads_sage_aep_{name_key}_{dt.datetime.now().strftime('%Y%m%dT%H%M')}"

step_client = boto3.client('stepfunctions')
step_client.start_execution(
    stateMachineArn = T1_VIZ_PIPELINE_ARN,
    name = pipeline_name,
    input= json.dumps(pipeline_input)
)

print(f"AEP : {flow_name} (yr) flows kicked off. Can take 15 - 45 mins.")
print(f"Step Function Pipeline : hv-vpp-ti-viz-pipeline : Pipeline run name started: {pipeline_name}")


In [ ]:

#### 25 Year Flow for Alaska

name_key = "25_ak"
flow_name = "25"
is_ak = True
pipeline_input = get_aep_pipeline_input(flow_name, is_ak)

# notice, slightly different object name
pipeline_name = f"hv_ti_data_loads_sage_aep_{name_key}_{dt.datetime.now().strftime('%Y%m%dT%H%M')}"

step_client = boto3.client('stepfunctions')
step_client.start_execution(
    stateMachineArn = T1_VIZ_PIPELINE_ARN,
    name = pipeline_name,
    input= json.dumps(pipeline_input)
)

print(f"AEP : {flow_name} (yr) flows kicked off. Can take 15 - 45 mins.")
print(f"Step Function Pipeline : hv-vpp-ti-viz-pipeline : Pipeline run name started: {pipeline_name}")


In [ ]:

#### 50 Year Flow for Alaska

name_key = "50_ak"
flow_name = "50"
is_ak = True
pipeline_input = get_aep_pipeline_input(flow_name, is_ak)

# notice, slightly different object name
pipeline_name = f"hv_ti_data_loads_sage_aep_{name_key}_{dt.datetime.now().strftime('%Y%m%dT%H%M')}"

step_client = boto3.client('stepfunctions')
step_client.start_execution(
    stateMachineArn = T1_VIZ_PIPELINE_ARN,
    name = pipeline_name,
    input= json.dumps(pipeline_input)
)

print(f"AEP : {flow_name} (yr) flows kicked off. Can take 15 - 45 mins.")
print(f"Step Function Pipeline : hv-vpp-ti-viz-pipeline : Pipeline run name started: {pipeline_name}")


In [ ]:

#### HW (High Water) Flow
name_key = "hw_ak"
flow_name = "high_water"
is_ak = True
pipeline_input = get_aep_pipeline_input(flow_name, is_ak)

# notice, slightly different object name
pipeline_name = f"hv_ti_data_loads_sage_aep_{name_key}_{dt.datetime.now().strftime('%Y%m%dT%H%M')}"

step_client = boto3.client('stepfunctions')
step_client.start_execution(
     stateMachineArn = T1_VIZ_PIPELINE_ARN,
     name = pipeline_name,
     input= json.dumps(pipeline_input)
)

print(f"AEP : {flow_name} (yr) flows kicked off. Can take 15 - 45 mins.")
print(f"Step Function Pipeline : hv-vpp-ti-viz-pipeline : Pipeline run name started: {pipeline_name}")
print("")

<h3>IMPORTANT: Return hv-vpp-ti-viz-fim-data-prep Lambda memory to 1,024 MiB</h3>

https://us-east-1.console.aws.amazon.com/lambda/home?region=us-east-1#/functions/hv-vpp-ti-viz-fim-data-prep?tab=code

Lambda name: hv-vpp-ti-viz-hand-fim-processing


<h2>6 - RUN CATCHMENT WORKFLOWS 2 CONFIGS AT A TIME. CHECK FOR STEP FUNCTION FINISHING BEFORE STARTING NEW ONE</h2>

In [ ]:

def get_catchment_pipepline_input(branch_key):

    catchment_name_key = f"catchments_{branch_key}_branches"

    pipeline_input = {
      "configuration": "reference",
      "job_type": "auto",
      "data_type": "channel",
      "keep_raw": False,
      "reference_time": dt.datetime.now().strftime('%Y-%m-%d 00:00:00'),
      "configuration_data_flow": {
        "db_max_flows": [],
        "db_ingest_groups": [],
        "python_preprocessing": []
      },
      "pipeline_products": [
        {
          "product": f"static_hand_{catchment_name_key}",
          "domain": "conus",
          "configuration": "reference",
          "product_type": "fim",
          "run": True,
          "fim_configs": [
            {
              "name": f"{catchment_name_key}",
              "target_table": f"fim_catchments.{catchment_name_key}",
              "fim_type": "hand",
              "sql_file": f"{catchment_name_key}"
            }
          ],
          "services": [
            f"static_hand_{catchment_name_key}_noaa"
          ],
          "raster_outputs": {
            "output_bucket": "",
            "output_raster_workspaces": []
          },
          "postprocess_sql": [],
          "product_summaries": [],
          "python_preprocesing_dependent": False
        }
        ,
        {
          "product": f"static_hand_{catchment_name_key}_hi",
          "domain": "conus",
          "configuration": "reference",
          "product_type": "fim",
          "run": True,
          "fim_configs": [
            {
              "name": f"{catchment_name_key}_hi",
              "target_table": f"fim_catchments.{catchment_name_key}_hi",
              "fim_type": "hand",
              "sql_file": f"{catchment_name_key}_hi"
            }
          ],
          "services": [
            f"static_hand_{catchment_name_key}_hi_noaa"
          ],
          "raster_outputs": {
            "output_bucket": "",
            "output_raster_workspaces": []
          },
          "postprocess_sql": [],
          "product_summaries": [],
          "python_preprocesing_dependent": False
        },
        {
          "product": f"static_hand_{catchment_name_key}_prvi",
          "domain": "conus",
          "configuration": "reference",
          "product_type": "fim",
          "run": True,
          "fim_configs": [
            {
              "name": f"{catchment_name_key}_prvi",
              "target_table": f"fim_catchments.{catchment_name_key}_prvi",
              "fim_type": "hand",
              "sql_file": f"{catchment_name_key}_prvi"
            }
          ],
          "services": [
            f"static_hand_{catchment_name_key}_prvi_noaa"
          ],
          "raster_outputs": {
            "output_bucket": "",
            "output_raster_workspaces": []
          },
          "postprocess_sql": [],
          "product_summaries": [],
          "python_preprocesing_dependent": False
        }
      ],
      "sql_rename_dict": {},
      "logging_info": {
          "Timestamp": int(dt.datetime.now().timestamp())
      }
    }

    return pipeline_input

print("get_catchment_pipepline_input loaded")


### 6a - Branch 0 Catchments. Wait until it is done before kicking off the next GMS (Level Path) catchments load a bit lower. ###

In [ ]:
# Note: The three db's here were renamed from:
# "branch_0_catchments", "branch_0_catchments_hi", "branch_0_catchments_prvi",

## IMPORTANT
'''
SUPER IMPORTANT:
Update hv-vpp-ti-viz-hand-fim-processing

    for memory and ephmeral
    
See Details in Step 3
'''

sf.execute_sql('''
TRUNCATE 
    fim_catchments.catchments_0_branches,
    fim_catchments.catchments_0_branches_hi,
    fim_catchments.catchments_0_branches_prvi;
''', db_type="egis")

# sf.execute_sql('''
# TRUNCATE 
#     fim_catchments.catchments_0_branches;
# ''', db_type="egis")

print("All Branch 0 Catchment tables truncated")
print("")

pipeline_name = f"hv_ti_data_loads_catchments_branch_0_{dt.datetime.now().strftime('%Y%m%dT%H%M')}"

pipeline_input = get_catchment_pipepline_input("0")
# print(pipeline_input)

step_client = boto3.client('stepfunctions')
step_client.start_execution(
    stateMachineArn=T1_VIZ_PIPELINE_ARN,
    name=pipeline_name,
    input=json.dumps(pipeline_input)
)

print("Catchments Branch 0 load kicked off. Takes appx 25 mins (depending on other processess)")
print(f"Step Function Pipeline : hv-vpp-ti-viz-pipeline : Run Name - {pipeline_name}")

### 6b - GMS (Level Paths / non branch 0) catchments ###

In [ ]:
sf.execute_sql('''
TRUNCATE
    fim_catchments.catchments_gms_branches,
    fim_catchments.catchments_gms_branches_hi,
    fim_catchments.catchments_gms_branches_prvi;
''', db_type="egis")

print("All gms / Level Path Branch Catchment tables truncated")

pipeline_name = f"hv_ti_data_loads_catchments_gms_{dt.datetime.now().strftime('%Y%m%dT%H%M')}"
pipeline_input = get_catchment_pipepline_input("gms")
# print(pipeline_input)

step_client = boto3.client('stepfunctions')
step_client.start_execution(
    stateMachineArn=T1_VIZ_PIPELINE_ARN,
    name=pipeline_name,
    input=json.dumps(pipeline_input)
)

print("Catchments GMS Branches (Level Paths / non branch 0) load kicked off."
      " Takes appx 25 mins (depending on other processess)")
print(f" Step Function Pipeline : hv-vpp-ti-viz-pipeline : Run Name - {pipeline_name}")


<h2>7 - Recreate derived.usgs_elev_table</h2>

In [ ]:

# Has appx 2,150 HUCs to process, but this section goes quickly.


print("usgs_elev_tables reload - started")
start_dt = dt.datetime.now()
print(f"Started: {dt.datetime.now().strftime('%m/%d/%Y, %H:%M:%S')}")

sf.execute_sql('DROP TABLE IF EXISTS derived.usgs_elev_table;')

uet_usecols = ['location_id', 'HydroID', 'dem_adj_elevation', 'nws_lid', 'levpa_id']

db_engine = sf.get_db_engine('viz')
s3_client = boto3.client("s3")
paginator = s3_client.get_paginator('list_objects')
operation_parameters = {'Bucket': TI_VIZ_DEPLOYMENT_FIM_BUCKET,
                        'Prefix': f'{HAND_DATASETS_DPATH}/',
                        'Delimiter': '/'}
page_iterator = paginator.paginate(**operation_parameters)
page_count = 0

for page in page_iterator:
    
    prefix_objects = page['CommonPrefixes']
    for i, prefix_obj in enumerate(prefix_objects):
        rec_num  = (i + 1) + (1000 * page_count)
        display_dt = dt.datetime.now().strftime("%m/%d/%Y, %H:%M:%S")
        print(f"Processing rec number {rec_num} : "
              f" On page  {page_count + 1} (appx 2150 recs in total) : {display_dt}")
        huc_prefix = prefix_obj.get("Prefix")
        usgs_elev_table_key = f'{huc_prefix}usgs_elev_table.csv'
        try:
            uet = s3_client.get_object(
                Bucket=TI_VIZ_DEPLOYMENT_FIM_BUCKET, 
                Key=usgs_elev_table_key
            )['Body']
            uet_df = pd.read_csv(uet, header=0, usecols=uet_usecols)
            # uet_df['fim_version'] = PUBLIC_FIM_VERSION
            uet_df[COLUMN_NAME_MODEL_VERSION] = HAND_MODEL_VERSION
            uet_df.to_sql(
                con=db_engine,
                dtype={
                    "location_id": Text(),
                    "nws_data_huc": Text()
                },
                schema='derived',
                name='usgs_elev_table',
                index=False, 
                if_exists='append'
            )
        except Exception as e:
            if "NoSuchKey" in str(e):
                pass
            else:
                raise e

    page_count += 1

end_dt = dt.datetime.now()
time_duration = end_dt - start_dt
print("***************")
print("usgs_elev_tables reload done")
print(f"Ended: {dt.datetime.now().strftime('%m/%d/%Y, %H:%M:%S')}")
print(f"... duration was  {str(time_duration).split('.')[0]}")
print("")

# Takes appx 3 mins to run


<h2>8 - Recreate derived.hydrotable_staggered</h2>

In [ ]:

# ++++++++++++++
# For 4.5.11.1 loads, it only loaded against hydrotable branch 0 which is likely a bug. 
# It affects hydrotable_staggered which affects a number of src_skill service but 
# might be in error with it loading just branch 0

# FOR 4.6.1.4, we loaded all hydrotables but against the huc level and not the branch level

# ++++++++++++++

# The Hydrotable is most a temp table (for now), but this will likely change
# if the forecast loads change. We will leave it for now.

# Based on the the HydroTable, we make the Hydrotable_staggered which is used in a service

print("hydrotable reloaded - started (est to take appx 8 hours")
start_dt = dt.datetime.now()
print(f"Started: {dt.datetime.now().strftime('%m/%d/%Y, %H:%M:%S')}")

# Drop them both as the staggered gets reloaded after the hydrotable
sf.execute_sql('DROP TABLE IF EXISTS derived.hydrotable;')
sf.execute_sql('DROP TABLE IF EXISTS derived.hydrotable_staggered;')

# sql = '''
# SELECT distinct LPAD(huc8::text, 8, '0') as huc8 FROM derived.featureid_huc_crosswalk WHERE huc8 is not null;
# '''
# df = sf.sql_to_dataframe(sql)

# Now loading the hydrotable
schema_name = "derived"
table_name = "hydrotable"
db_engine_key = "viz"
full_s3_path = f"s3://{TI_VIZ_DEPLOYMENT_FIM_BUCKET}/{HAND_DATASETS_DPATH}"
file_name = "hydrotable.csv"

ht_usecols = ['HydroID', 'feature_id', 'stage', 'discharge_cms']

print(f"Downloading from {full_s3_path}")

file_list = s3_sf.get_s3_subfolder_file_names(TI_VIZ_DEPLOYMENT_FIM_BUCKET, HAND_DATASETS_DPATH,
                                              file_name, False, False)

num_files = len(file_list)
if num_files == 0:
    raise Exception(f"Unable to find any records with the name of {file_name} in {full_s3_path}")

db_engine = sf.get_db_engine(db_engine_key)
s3_client = boto3.client("s3")

page_count = 0
for i, s3_key_path in enumerate(file_list):
    # s3_file_path = f"s3://{TI_VIZ_DEPLOYMENT_FIM_BUCKET}/{s3_key_path}"
    print(f"Loading {i + 1} of {num_files} : {s3_key_path}")

    ht = s3_client.get_object(
        Bucket=TI_VIZ_DEPLOYMENT_FIM_BUCKET,
        Key=s3_key_path
    )['Body']
    # print("... Reading with pandas ...")
    ht_df = pd.read_csv(ht, header=0, usecols=ht_usecols)
    ht = None

    # print("... saving dataframe to db")
    ht_df[COLUMN_NAME_MODEL_VERSION] = HAND_MODEL_VERSION
    ht_df.to_sql(
        con=db_engine,
        schema=schema_name,
        name=table_name,
        index=False,
        if_exists='append',
        method='multi'
    )
    ht_df = None
    # print("... saved to db")

end_dt = dt.datetime.now()
time_duration = end_dt - start_dt
print("***************")
print("hydrotable reload done")
print(f"Ended: {dt.datetime.now().strftime('%m/%d/%Y, %H:%M:%S')}")
print(f"... duration was  {str(time_duration).split('.')[0]}")
print("")

# takes only a few mins

print("hydrotable_staggered started")

start_dt = dt.datetime.now()
print("")

sql = '''
DROP TABLE IF EXISTS derived.hydrotable_staggered;
SELECT
    et.location_id,
    ht.feature_id,
    (stage + et.dem_adj_elevation) * 3.28084 as elevation_ft,
    LEAD((stage + et.dem_adj_elevation) * 3.28084) OVER (PARTITION BY ht.feature_id ORDER BY ht.feature_id, stage) as next_elevation_ft,
    discharge_cms * 35.3147 as discharge_cfs,
    LEAD(discharge_cms * 35.3147) OVER (PARTITION BY ht.feature_id ORDER BY ht.feature_id, stage) as next_discharge_cfs
INTO derived.hydrotable_staggered
FROM derived.hydrotable AS ht
JOIN derived.usgs_elev_table AS et ON ht."HydroID" = et."HydroID" AND et.location_id IS NOT NULL;
'''
sf.execute_sql(sql)

print("hydrotable_staggered reload done")
end_dt = dt.datetime.now()
time_duration = end_dt - start_dt
print(f"... staggered duration was  {str(time_duration).split('.')[0]}")



In [ ]:

# we don't need the hydrotable anymore as it has been reloaded and adjusted above in hydrotable_staggered
# sf.execute_sql('DROP TABLE IF EXISTS derived.hydrotable;')
# print("Done dropping derived.hydrotable, post hydrotable_staggered load")

# Apr 11, 2025.. let's keep it for now for this release.


<h2>9 - Recreate derived.usgs_rating_curves_staggered</h2>

In [ ]:
# run the script to load the usgs_rating_curve.csv.

# derived.usgs_rating_curves is really a temp table and does not need to be kept at this time.
# we keep and distrbute derived.usgs_rating_curves_staggered

# takes appx 

sql = '''
    DROP TABLE IF EXISTS derived.usgs_rating_curves;
    DROP TABLE IF EXISTS derived.usgs_rating_curves_staggered;
'''
sf.execute_sql(sql)

print("Done dropping usgs_rating_curves and usgs_rating_curves_staggered")

print("Starting loading usgs_rating_curves. Takes appx 30 mins.")
print("Don't worry about the pandas warning.. it is running.")

start_dt = dt.datetime.now()
ref_time = dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"ref_time is {ref_time}")
event = {
    'target_table': 'derived.usgs_rating_curves',
    'target_cols': ['location_id', 'flow', 'stage', 'navd88_datum', 'elevation_navd88'],
    'file': f'{QA_DATASETS_DPATH}/usgs_rating_curves.csv',
    'bucket': TI_VIZ_DEPLOYMENT_FIM_BUCKET,
    'reference_time': f'{ref_time}',
    'keep_flows_at_or_above': 0,
    'iteration_index': 0
}

execute_db_ingest(event, None)

print("done loading usgs_rating_curves")
end_dt = dt.datetime.now()
time_duration = end_dt - start_dt
print(f"... duration was  {str(time_duration).split('.')[0]}")


# Takes under a minute
print("Starting usgs_rating_curves_staggered build based on usgs_rating_curve table")

sql = '''
SELECT 
    location_id,
    flow as discharge_cfs, 
    LEAD(flow) OVER (PARTITION BY location_id ORDER BY location_id, stage) as next_discharge_cfs,
    stage,
    navd88_datum,
    elevation_navd88 as elevation_ft,
    LEAD(elevation_navd88) OVER (PARTITION BY location_id ORDER BY location_id, stage) as next_elevation_ft
INTO derived.usgs_rating_curves_staggered
FROM derived.usgs_rating_curves;
'''
sf.execute_sql(sql)

print("Done loading usgs_rating_curves_staggered")


In [ ]:

# usgs_rating_curves is a temp table and is loaded with some changes into the usgs_rating_curves_staggered
sf.execute_sql('DROP TABLE IF EXISTS derived.usgs_rating_curves;')
print("Done dropping derived.usgs_rating_curves, post loading usgs_rating_curves_staggered")

<h2>10 - UPDATE SRC SKILL METRICS IN DB</h2>

In [ ]:

# Load the src_skill_temp table
start_dt = dt.datetime.now()

sf.eecute_sql('DROP TABLE IF EXISTS derived.src_skill_temp;', db_type='viz')
sf.execute_sql('DROP TABLE IF EXISTS derived.src_skill;', db_type='viz')

file_handle = 'agg_nwm_recurr_flow_elev_stats_location_id.csv'

print("Reading file...")
# df = pd.read_csv(local_download_path)
file_to_download = f"{MISC_FIM_DATASETS_DPATH}/{file_handle}"
df = s3_sf.download_S3_csv_files_to_df_from_list(TI_VIZ_DEPLOYMENT_FIM_BUCKET, [file_to_download], True)
print(f"File read. {len(df)} records found")

db_type = "viz"
db_engine = sf.get_db_engine(db_type)

df.to_sql(
    name='src_skill_temp',
    con=db_engine,
    schema='derived',
    if_exists='replace',
    index=False
)

print("Done loading derived.src_skill_temp table")
end_dt = dt.datetime.now()
time_duration = end_dt - start_dt
print(f"... duration was  {str(time_duration).split('.')[0]}")


In [ ]:

# Load into src_skill table adding geometry to it from external.usgs_gage. Yes.. more/less straight from WRDS tables
# Some recs appear to be in error in the csv. location id = 394220106431500 (those are dropped below)

# TODO: needs to be run.


start_dt = dt.datetime.now()

sf.execute_sql('DROP TABLE IF EXISTS derived.src_skill;', db_type='viz')
sf.execute_sql('DROP TABLE IF EXISTS reference.src_skill;', db_type='egis')

sql = f'''
SELECT
    (row_number() OVER ())::int as oid,
    gage.name,
    LPAD(skill_temp.location_id::text, 8, '0') as location_id,
    skill_temp.nrmse,
    skill_temp.mean_abs_y_diff_ft,
    skill_temp.mean_y_diff_ft,
    skill_temp.percent_bias,
    '{HAND_MODEL_VERSION}' as {COLUMN_NAME_MODEL_VERSION},
    gage.geo_point as geom
INTO derived.src_skill
FROM derived.src_skill_temp skill_temp
JOIN external.usgs_gage AS gage ON LPAD(gage.usgs_gage_id::text, 8, '0') = LPAD(skill_temp.location_id::text, 8, '0')
'''

sf.execute_sql(sql)

print("Done loading derived.src_skill table")
end_dt = dt.datetime.now()
time_duration = end_dt - start_dt
print(f"... duration was  {str(time_duration).split('.')[0]}")

print("Now moving from derived.src_skill to the reference schema")
sf.move_data_from_viz_to_egis("derived.src_skill", "reference.src_skill")
print("Done")


<h2>11 - UPDATE FIM PERFORMANCE METRICS IN DB</h2>

In [ ]:
# clean up tables for new load

table_names = [
    "reference.fim_performance_points",
    "reference.fim_performance_polys",
    "reference.fim_performance_catchments"
]

for tb_name in table_names:
    sql = f"TRUNCATE TABLE {tb_name}"
#    print(sql)
    sf.execute_sql(sql,db_type='egis')


print(f"All fim_performance tables truncated if they exist")


In [ ]:

# Load the new fim performance tables 

print("FIM performance data reload - started (takes appx 20 mins for all three combined)")
start_dt = dt.datetime.now()
print(f"Started: {dt.datetime.now().strftime('%m/%d/%Y, %H:%M:%S')}")

# file_handles = ['fim_performance_points.csv']
file_handles = ['fim_performance_points.csv', 'fim_performance_polys.csv', 'fim_performance_catchments.csv']
# file_handles = ['fim_performance_points.csv', 'fim_performance_polys.csv']
# file_handles = ['fim_performance_catchments.csv']

db_engine = sf.get_db_engine('egis')

# Apr 28, 2025: Added this part of cols_to_drop after running this set for v2.2
# We can check it on the next one
# and re-eval what we are doing with the oid column
# many of our tools are not loading an oid column but adding it after the table
# is loaded to ensure uniqueness
cols_to_drop = [:"oid", "version"]

for file_handle in file_handles:

    s3_file_key = f"{QA_DATASETS_DPATH}/{file_handle}"

    df = s3_sf.load_S3_csv_to_df(TI_VIZ_DEPLOYMENT_FIM_BUCKET, s3_file_key,
                                 TI_ACCESS_KEY, TI_SECRET_KEY, TI_TOKEN, True)

    print("")
    if len(df) == 0:
        raise Exception(f"no records found for file: {file_handle}. Check file path and/or aws creds / session keys")
        break
    else:
        print(f"loading {len(df)} records")
    
    # drop it for now, we will rebuild it
    # if 'oid' in df.columns:
    #     df = df.drop('oid', axis=1)

    df = df.drop(cols_to_drop, errors='ignore', axis=1)

    # Rename headers.
    if file_handle == 'fim_performance_points.csv':
        df = df.rename(columns={'Unnamed: 0': 'oid', 'geometry': 'geom'})
    else:
        df = df.rename(columns={'Unnamed: 0': 'oid', 'geometry': 'geom', 'huc': 'huc8'})

    # print(df.dtypes)
    # Convert all field names to lowercase (needed for ArcGIS Pro).
    df.columns = df.columns.str.lower()

    # Enforce data types on df before loading in DB (TODO: need to create special cases for each layer).
    if file_handle == 'fim_performance_points.csv':
        df = df.astype({'huc': 'str'})
    else:
        df = df.astype({'huc8': 'str'})
    df = df.fillna(0)
    try:
        df = df.astype({'feature_id': 'int'})
        df = df.astype({'feature_id': 'str'})
        df = df.astype({'oid': 'int'})
    except KeyError:  # If there is no feature_id field
        pass
    try:
        df = df.astype({'nwm_seg': 'int'})
        df = df.astype({'nwm_seg': 'str'})
    except KeyError:  # If there is no nwm_seg field
        pass
    try:
        df = df.astype({'usgs_gage': 'int'})
        df = df.astype({'usgs_gage': 'str'})
    except KeyError:  # If there is no usgs_gage field
        pass

    # zfill HUC8 field.
    if file_handle == 'fim_performance_points.csv':
        df['huc'] = df['huc'].apply(lambda x: x.zfill(8))
    else:
        df['huc8'] = df['huc8'].apply(lambda x: x.zfill(8))

    # add model version column
    df[COLUMN_NAME_FIM_VERSION] = PUBLIC_FIM_VERSION
    df[COLUMN_NAME_MODEL_VERSION] = HAND_MODEL_VERSION

    # if "version" in df:
    #     df = df.drop("version", axis=1)

    # Upload df to database.
    stripped_layer_name = file_handle.replace(".csv", "")
    table_name = "reference." + stripped_layer_name
    print("... Loading data into DB")

    # Chunk load data into DB

    if file_handle in ['fim_performance_catchments.csv']:

        # Create list of df chunks
        n = 10000  # chunk row size
        list_df = [df[i:i+n] for i in range(0,df.shape[0],n)]
        # geometry = 'MULTIPOLYGON'
        # Load the first chunk into the DB as a new table
        first_chunk_df = list_df[0]
        # print(first_chunk_df.shape[0])

        num_of_chunks = len(list_df)
        print(f"Loading in {num_of_chunks} chunks")
        
        print(f"Loading first chunk of {num_of_chunks} ")
        first_chunk_df.to_sql(
            name=stripped_layer_name,
            con=db_engine,
            schema='reference',
            if_exists='replace',
            index=False,
            dtype={'oid': sqlalchemy.types.Integer(),
                   'version': sqlalchemy.types.String(),
                   'geom': Geometry('MULTIPOLYGON', srid=HYDROVIS_CRS_NUMBER)
                  }
        )
        # Load remaining chunks into newly created table

        for i, remaining_chunk_df in enumerate(list_df[1:]):
            print(f"Loading {i} chunk of {num_of_chunks} ")
            # print(remaining_chunk_df.shape[0])
            remaining_chunk_df.to_sql(
                name=stripped_layer_name,
                con=db_engine,
                schema='reference',
                if_exists='append',
                index=False,
                dtype={'version': sqlalchemy.types.String(),
                       'geom': Geometry('MULTIPOLYGON', srid=HYDROVIS_CRS_NUMBER)
                      }
            )
    else:
        if 'points' in stripped_layer_name: geometry='POINT'
        if 'polys' in stripped_layer_name: geometry='POLYGON'
        # print("GEOMETRY")
        # print(geometry)
        df.to_sql(
            name=stripped_layer_name,
            con=db_engine,
            schema='reference',
            if_exists='replace',
            index=False,
            dtype={'oid': sqlalchemy.types.Integer(),
                   'version': sqlalchemy.types.String(),
                   'geom': Geometry(geometry, srid=HYDROVIS_CRS_NUMBER)
                  }
        )

    print(f"... >>> {file_handle} downloaded and loaded")
    print("")

end_dt = dt.datetime.now()
time_duration = end_dt - start_dt
print("FIM Performance files loaded done")
print(f"Ended: {dt.datetime.now().strftime('%m/%d/%Y, %H:%M:%S')}")
print(f"... duration was  {str(time_duration).split('.')[0]}")


<h2>12 - CatFIM (Stage-Based and Flow-Based)</h2>

<h4>Function to load CatFIM Data (Non Public / FIM 30)</h4>

In [ ]:
''' Function to load CatFIM data (for any flow / stage / library / sites but not the public/fim 30)'''


# Update: Apr 28, 2025: We have data comign in for fid / unnamed and we turn it into an oid column, but there is dups
# Drop those columns and make our own oid after the table is loaded.

# ****** needs update for 2.2 / 5.2 See https://github.com/NOAA-OWP/hydrovis/issues/1028
def load_catfim_table(catfim_type):

    '''
    Inputs:
        - catfim_type: name identififer for the set, such as "flow_based_catfim_library" or "flow_based_catfim_sites", etc
              Sometimes the file_handle name can be the name of the s3 file (without extension) and/or the table
              name.
    '''

    # Will not error out if the column exists
    # Ee will rebuild the oid after the fact
    cols_to_drop = ["oid", "unnamed: 0", "fid", "viz"]

    bp_db_engine_key = "egis"

    # --------------------------------------
    # Drop the original Db if already in place
    table_name = catfim_type  # yes, dup variable for now

    if catfim_type in ['flow_based_catfim_library', 'stage_based_catfim_library']:  # Libraries
        table_name = table_name.replace("_library", "")

    sf.execute_sql(f"DROP TABLE IF EXISTS reference.{table_name};", db_type=bp_db_engine_key)
    print(f"Dropping reference.{table_name} table if it existed")
    print("")

    # --------------------------------------
    # Get the data from S3 and load it into a df
    file_to_download = f"{QA_DATASETS_DPATH}/{catfim_type}.csv"

    # print(f"Downloading {file_to_download} ... ")

    df = s3_sf.load_S3_csv_to_df(TI_VIZ_DEPLOYMENT_FIM_BUCKET, file_to_download,
                                 TI_ACCESS_KEY, TI_SECRET_KEY, TI_TOKEN, True)
    num_recs = len(df)

    if num_recs == 0:
        raise Exception(f"no records found for file: {file_to_download}. Check file path and/or aws creds / session keys")

    print(f"File read. {num_recs} records to load")

    # --------------------------------------
    # Adjusting Columns and data
    # Rename headers. All files this name

    # Convert all field names to lowercase (needed for ArcGIS Pro).
    df.columns = df.columns.str.lower()

    # Will not error out if the column exists
    # We will rebuild the oid after the fact
    df = df.drop(cols_to_drop, errors='ignore', axis=1)

    df = df.rename(columns={'geometry': 'geom'})

    if "huc" in df.columns:
        df = df.rename(columns={'huc': 'huc8'})

    # Enforce data types on df before loading in DB (TODO: need to create special cases for each layer).
    df = df.astype({'huc8': 'str'})
    df = df.fillna(0)
    try:
        df = df.astype({'feature_id': 'int'})
        df = df.astype({'feature_id': 'str'})
    except KeyError:  # If there is no feature_id field
        pass
    try:
        df = df.astype({'nwm_seg': 'int'})
        df = df.astype({'nwm_seg': 'str'})
    except KeyError:  # If there is no nwm_seg field
        pass
    try:
        df = df.astype({'usgs_gage': 'int'})
        df = df.astype({'usgs_gage': 'str'})
    except KeyError:  # If there is no usgs_gage field
        pass

    # zfill HUC8 field.
    df['huc8'] = df['huc8'].apply(lambda x: x.zfill(8))

    if '_sites' in catfim_type:
        df = df.astype({'nws_data_rfc_forecast_point': 'str'})
        df = df.astype({'nws_data_rfc_defined_fcst_point': 'str'})
        df = df.astype({'nws_data_riverpoint': 'str'})

    # add fim_version field
    df[COLUMN_NAME_FIM_VERSION] = PUBLIC_FIM_VERSION
    # As of Apr 2025, the source gpkgs have a model_version column in them, but we will override
    # to our column value here.
    df[COLUMN_NAME_MODEL_VERSION] = HAND_MODEL_VERSION

    # Apr 2025: Catfim source does have a new column named product_version
    # (CatFiM 2_2). We will load it to the new HV tables but not include it in the mapx

    # Apr 2025: CatFIM has a new field named "is_interval", but for stage library only
    # and loaded but not used.

    # --------------------------------------
    # Load to DB
    # Chunk load data into DB

    n = 10000  # chunk row size  (number of rows at a time, each row is pretty big)
    # the number of rows we send in depend on how much horsepower is behind this script

    db_engine = sf.get_db_engine(bp_db_engine_key)

    if catfim_type in ['flow_based_catfim_library', 'stage_based_catfim_library']:  # Libraries

        # Note: to_sql also has a "chunksize" attribute as well, but by doing it ourselves
        # we can see progress
        print(f"Chunk loading... into {table_name} -- {n} records at a time")
        print("")
        chunk_df = [df[i:i+n] for i in range(0, df.shape[0], n)]

        # Load the first chunk into the DB as a new table
        first_chunk_df = chunk_df[0]
        num_chunks = len(chunk_df)

        print(f" ... loading chunk 1 of {num_chunks}")

        first_chunk_df.to_sql(
            name=table_name,
            con=db_engine,
            schema='reference',
            if_exists='replace',
            index=False,
            dtype={'geom': Geometry('MULTIPOLYGON', srid=HYDROVIS_CRS_NUMBER)}
        )

        # Load remaining chunks into newly created table
        ctr = 1  # Already loaded one
        for remaining_chunk in chunk_df[1:]:
            # print(remaining_chunk.shape[0])
            ctr += 1
            print(f" ... loading chunk {ctr} of {num_chunks}")
            remaining_chunk.to_sql(
                        name=table_name,
                        con=db_engine,
                        schema='reference',
                        if_exists='append',
                        index=False,
                        dtype={'geom': Geometry('MULTIPOLYGON', srid=HYDROVIS_CRS_NUMBER)}
                    )
        # end for
    else:  # sites tables
        print(f"Loading data into {table_name} ...")

        df.to_sql(
            name=table_name,
            con=db_engine,
            schema='reference',
            if_exists='replace',
            index=False,
            dtype={'geom': Geometry('POINT', srid=HYDROVIS_CRS_NUMBER)})

    # This should auto create a gist index against the geometry column
    # if that index name already exists, the upload will fail, the index can not pre-exist
    # Best to drop the table before loading.

    # add an oid column back in
    print("Adding oid key")
    sql = f"ALTER TABLE reference.{table_name} ADD COLUMN OID SERIAL PRIMARY KEY;"
    sf.execute_sql(sql, db_type=bp_db_engine_key)

    # return

print("load_catfim_table function loaded")


<h3>12.a - Updated Flow and Stage Based CatFIM Data (Non Public/ FIM 30)</h3>

<h3>AUG 2024: IMPORTANT NOTE:</h3>
The stage based catfim (library) csv has grown to appx 10 GiB. Our current notebook, hv-vpp-ti-viz-notebook only has 15 GiB memory.
Running tool can easily overwhelm the notebook server and freeze it up forcing a reboot.
Sometimes when the notebook instance comes back up, it no longer has ths swap system in place. You will need most of the memory
and some swap to load it.  Keep an eye a "terminal" windows and keep entering `free -h` to keep an eye on it's usage.
</br>
We will need to review to see if we want to:

1. Upgrade this notebook server with more memory (and harddrive space would be good)

2. Change the load of the catfim library (non sites) data to another system. Maybe we can load it via a lambda to an EC2 or something?

3. Get the FIM Team to break it to smaller pieces, but watch carefully for the OID system (unique id for all records)

**When you are done running this script, Please restart this kernal as it does not appear to be releasing all memory. (memory leak?)**

<H3>Update Apr 2025:</h3>
When you run this now on the large sagemaker notebook, it can handle alot more, including a higher chunk size.

Eventually, this should become a lambda 

In [ ]:

print("Starting of CatFIM data")

# the two sites csv's take a few seconds to laod
# Stage based library takes appx 7 to 14 mins (at 1,000 chunks small sagemaker)
# Stage based library takes appx 13 mins (at 10,000 chunks small sagemaker)
# flow based library takes apppx 10 secs

# catfim_types = ['flow_based_catfim_sites']
# catfim_types = ['flow_based_catfim_library']
# catfim_types = ['flow_based_catfim_library', 'flow_based_catfim_sites']
# catfim_types =  ['stage_based_catfim_library', 'stage_based_catfim_sites']
catfim_types = ['stage_based_catfim_sites']
# catfim_types = ['stage_based_catfim_library']

start_dt = dt.datetime.now()

for catfim_type in catfim_types:
    print(f"Loading {catfim_type} data")
    load_catfim_table(catfim_type)

print("")
end_dt = dt.datetime.now()
time_duration = end_dt - start_dt
print(f"... duration was  {str(time_duration).split('.')[0]}")
# load in just a minute


<h3>12.b - Load CatFIM "public" FIM 30 DBs</h3>

In [ ]:

# Dec 2024: We don't use the flow based public db's
#  but the tables need to be removed and the services too, code, and published services
# Apr 2025: Trace the tables names through services and enviros, might not be loading the right tables

print("Loading CatFIM Public datasets (FIM 30)")

catfim_types = ["stage_based_catfim", "stage_based_catfim_sites"]

__public_fim_release = "fim_30"  # The new fim public release being loaded (ie. fim_10, fim_30, fim_60..)

start_dt = dt.datetime.now()

for catfim_type in catfim_types:
    print("")
    sql = f'''
    DROP TABLE IF EXISTS reference.{catfim_type}_{__public_fim_release};

    SELECT
        catfim.*,
        '{__public_fim_release}' as public_fim_release
    INTO reference.{catfim_type}_{__public_fim_release}
    FROM reference.{catfim_type} as catfim
    JOIN reference.public_fim_domain as fim_domain ON ST_Intersects(catfim.geom, fim_domain.geom)
    '''
    sf.execute_sql(sql, db_type='egis')
    print(f"public {__public_fim_release} data load for {catfim_type} is complete")

# what about indexes again?

# for db_name in db_names:
#     new_table_name = f"reference.{db_name}_{db_name_appendix}"
#     sql = f"CREATE TABLE IF NOT EXISTS {new_table_name} AS TABLE reference.{db_name}"
#     sf.execute_sql(sql, db_type='egis')
#     print(f"{db_name} copied to {new_table_name} if it does not already exist")

print("")
end_dt = dt.datetime.now()
time_duration = end_dt - start_dt
print(f"... duration was  {str(time_duration).split('.')[0]}")


<h2>13 - Load Bridge Flood Threat Data (BITS) and Bridge Centroids</h2>

In [ ]:
# Has appx 2,150 HUCs to process, but this section goes quickly.

# TODO: Apr 10, 2025 This should be moved to a step function

# Why is this different from the way others load? It is GPKGs. It is also at the HUC level.

# Independendant from BITS, we also need to relod bridge_centroids. We 
# smaller set of columns. We keep them seperate as one is static and on is dynamic

'''
# NOTES:
   - Loaded only the imperial data system (Ft, Miles, cfs) and the metric data.
   - We will load all columns, then drop the ones we don't want.
   - Loading at the HUC level. 

   - This is loading to viz.derived.bridge_points (bp) for use for dynamic flow processing
     AND loading egis.reference.bridge_centroids (bc) for the static bridge centroids layer
'''

print("Bridge Flood Threat data load started. Note: most HUCs have a bridge file but not all")
print("This will load both the flow based viz.derived.bridge_points data and the egis.reference.bridge_centroids data")

# Initial load of bridge_points
bp_schema_name = "derived"
bp_table_name = "bridge_points"
bp_db_engine_key = "viz"

# Initial load of bridge_points
bc_schema_name = "reference"
bc_table_name = "bridge_centroids"
bc_db_engine_key = "egis"

# drop tables before starting  (bridge_points first)
sf.execute_sql(f'DROP TABLE IF EXISTS {bp_schema_name}.{bp_table_name};', db_type=bp_db_engine_key)
# drop bridge centriods as well
sf.execute_sql(f'DROP TABLE IF EXISTS {bc_schema_name}.{bc_table_name};', db_type=bc_db_engine_key)

# TESTING
# full_s3_path = "s3://{TI_VIZ_DEPLOYMENT_FIM_BUCKET}/fim/hand_bridge_test/hand_datasets/"
full_s3_path = f"s3://{TI_VIZ_DEPLOYMENT_FIM_BUCKET}/{HAND_DATASETS_DPATH}"

file_name = "osm_bridge_centroids.gpkg"

# The threashold columns listed here are the metric ones we don't want.
# As this is a gpkg, we don't need to drop the fid
cols_to_drop = ["threshold_hand", "threshold_hand_75",
                "threshold_discharge", "threshold_discharge75",
                "order_", "huc10", "huc6", "crossing_feature_id"]

# bc_cols_to_keep = ["osmid", "name", "feature_id", "hydroid", "huc8",
#                    "bridge_type", "model_version", "geometry"]
bc_cols_to_keep = ["osmid", "name", "huc8", "bridge_type", "model_version", "geometry"]

print(f"Downloading from {full_s3_path}")
start_dt = dt.datetime.now()
print(f"Started: {start_dt.strftime('%m/%d/%Y, %H:%M:%S')}")

bridge_file_list = s3_sf.get_s3_subfolder_file_names(TI_VIZ_DEPLOYMENT_FIM_BUCKET, HAND_DATASETS_DPATH, file_name)

# TEST
# bridge_file_list = s3_sf.get_s3_subfolder_file_names(TI_VIZ_DEPLOYMENT_FIM_BUCKET, 
#                                                "fim/hand_bridge_test/hand_datasets",
#                                                file_name, False, False)

if len(bridge_file_list) == 0:
    raise Exception(f"Unable to find any records with the name of {file_name} in {full_s3_path}")

print("************")
num_files = len(bridge_file_list)
print(f"Number of bridge files found is {num_files}")
print("")

for i, s3_key_path in enumerate(bridge_file_list):
    s3_file_path = f"s3://{TI_VIZ_DEPLOYMENT_FIM_BUCKET}/{s3_key_path}"
    print(f"Loading {i + 1} of {num_files} : {s3_key_path}")
    # print(s3_file_path)
    bridge_df = gpd.read_file(s3_file_path,
                              engine="pyogrio",
                              use_arrow=True)

    # +++++++++++++
    # We will adjust what we need and load it to the bridge_points layer first
    # then lower, we will copy it to a new gdf which filters down to only a few columns
    # for use in bridge_centroids
    # --------------------------------------
    # Adjusting Columns and data

    # Convert all field names to lowercase (needed for ArcGIS Pro).
    bridge_df.columns = bridge_df.columns.str.lower()

    bridge_df["risk_status_code"] = -1
    # add model version column
    bridge_df[COLUMN_NAME_MODEL_VERSION] = HAND_MODEL_VERSION

    bridge_df = bridge_df.drop(cols_to_drop, errors='ignore', axis=1)
    bridge_df['name'] = bridge_df['name'].fillna("")

    adj_column_types = {'huc8': str,
                        'has_lidar_tif': str,
                        'is_backwater': str,
                        'mainstem': str}

    bridge_df = bridge_df.astype(adj_column_types)

    # Enforce data types on df before loading in DB (TODO: need to create special cases for each layer).
    bridge_df['huc8'] = bridge_df['huc8'].apply(lambda x: x.zfill(8))

    bridge_df['has_lidar_tif'] = bridge_df['has_lidar_tif'].apply(lambda x: "False" if x == "N" else "True")
    bridge_df['mainstem'] = bridge_df['mainstem'].apply(lambda x: "True" if x == "1" else "False")
    bridge_df['is_backwater'] = bridge_df['is_backwater'].apply(lambda x: "True" if x == "1" else "False")

    # bridge_df = bridge_df.rename(columns={'geometry': 'geom'}) # because we have a gdf not df

    # Reproject. If AK.. it comes in as 3338 and if CONUS it comes in as 5070
    # but now all are 3857
    reproj_bridge_df = bridge_df.to_crs(HYDROVIS_CRS)

    # +++++++++++++
    # Load the bridge_points db
    db_engine = sf.get_db_engine(bp_db_engine_key)
    reproj_bridge_df.to_postgis(
                con=db_engine,
                schema=bp_schema_name,
                name=bp_table_name,
                if_exists='append',
                dtype={'geom': Geometry('POINT', srid=HYDROVIS_CRS_NUMBER)}
        )

    # +++++++++++++
    # Now thin out to just the columns we need for bridge points.
    bridge_cents_df = reproj_bridge_df[bc_cols_to_keep]

    db_engine = sf.get_db_engine(bc_db_engine_key)
    bridge_cents_df.to_postgis(
                con=db_engine,
                schema=bc_schema_name,
                name=bc_table_name,
                if_exists='append',
                dtype={'geom': Geometry('POINT', srid=HYDROVIS_CRS_NUMBER)}
        )

print("++++++++++++")
print("Bridge data loaded to both tables")
print("")
print("Adding Indexes and updating the geom column")
print("Starting Bridge Points first")
sql = f"ALTER TABLE {bp_schema_name}.{bp_table_name} ADD COLUMN OID SERIAL PRIMARY KEY;"
sf.execute_sql(sql, db_type=bp_db_engine_key)
print("...")

# to_postgis needs the "geometry" column, but arc wants the column named as "geom"
# Did not work above to change the name as we were using a gdf not a df.
# It was loaded from a gpkg and not a csv
sql = f"ALTER TABLE {bp_schema_name}.{bp_table_name} RENAME geometry TO geom;"
sf.execute_sql(sql, db_type=bp_db_engine_key)
print("...")

print("Now Starting Bridge Centroids")
sql = f"ALTER TABLE {bc_schema_name}.{bc_table_name} ADD COLUMN OID SERIAL PRIMARY KEY;"
sf.execute_sql(sql, db_type=bc_db_engine_key)
print("...")

sql = f"ALTER TABLE {bc_schema_name}.{bc_table_name} RENAME geometry TO geom;"
sf.execute_sql(sql, db_type=bc_db_engine_key)
print("...")


print("***************")
print("Bridge Flood Threat data load completed")
end_dt = dt.datetime.now()
time_duration = end_dt - start_dt
print(f"Ended: {end_dt.strftime('%m/%d/%Y, %H:%M:%S')}")
print(f"... duration was  {str(time_duration).split('.')[0]}")
print("")

<h2>14 - HECRAS boundaries</h2>

In [5]:

# Note: This db needs to be in both derived (for dynamic processing) and reference for static referencing)

# TODO: This loads to reference. If needed for next version (FIM 6.0?), then add code to copy it from
# refernce to derived.

file_to_download = f"{S3_FIM_VERSION_PATH}/FIM_30_hecras_boundaries.csv"
table_name = "hecras_boundaries"
schema_name = "reference"
db_instance_name = "egis"

print("HECRAS Boundaries data load started")
print(f"Downloading from {file_to_download}")
start_dt = dt.datetime.now()

sf.execute_sql(f'DROP TABLE IF EXISTS {schema_name}.{table_name} CASCADE;', db_type=db_instance_name)

df = s3_sf.load_S3_csv_to_df(TI_VIZ_DEPLOYMENT_FIM_BUCKET, file_to_download,
                             TI_ACCESS_KEY, TI_SECRET_KEY, TI_TOKEN, True)
if len(df) == 0:
    raise Exception(f"no records found for file: {file_to_download}. Check file path and/or aws creds / session keys")

print(f"File read. {len(df)} records (HUCs) found")
# Convert all field names to lowercase (needed for ArcGIS Pro).
df.columns = df.columns.str.lower()

df = df.astype({'huc8': 'str'})
# zfill HUC8 field.
df['huc8'] = df['huc8'].apply(lambda x: x.zfill(8))

# is_active  (text True, False?)  - check what it looks liek when loaded

df = df.rename(columns={'unnamed: 0': 'oid',
                        'geometry': 'geom'})

db_engine = sf.get_db_engine(db_instance_name)

df.to_sql(
    name=table_name,
    con=db_engine,
    schema=schema_name,
    if_exists='replace',
    index=False,
    dtype={'oid': sqlalchemy.types.Integer(),
           'geom': Geometry('MULTIPOLYGON', srid=5070)}
)

print("")
print("HECRAS boundaries data load completed")
end_dt = dt.datetime.now()
time_duration = end_dt - start_dt
print(f"... duration was  {str(time_duration).split('.')[0]}")


HECRAS Boundaries data load started
.. Downloading: s3://hydrovis-ti-deployment-us-east-1/fim/v5_2/FIM_30_hecras_boundaries.csv
.. Downloading complete
File read. 434 records (HUCs) found

HECRAS boundaries data load completed
... duration was  0:00:09


<h2>15 - Clear the HAND Cache</h2>

sql = """
TRUNCATE TABLE handfim_cache.hydrotable_cached;
TRUNCATE TABLE handfim_cache.hydrotable_cached_geo;
TRUNCATE TABLE handfim_cache.hydrotable_cached_max;
TRUNCATE TABLE handfim_cache.hydrotable_cached_zero_stage;
"""
sf.execute_sql(sql)
print("caches cleared")

<h2>16 - SAVE TO REPO (AND REDEPLOY TO TI WITH NEW VERSION VARIABLE IN TERRAFORM ??)</h2>